# Extract Goals from Transcripts - Whole Transcript

This notebook proceeds to extract goals from interview transcripts in two stages: (1) generate goals over the whole transcript; (2) trace generated goals back to the source text in the transcript.

In [2]:
from openai import OpenAI

client = OpenAI()
gpt4o_model = "gpt-4o-2024-08-06"
gpt5_2_model = "gpt-5.2-2025-12-11"

def prompt_model(prompt, model=gpt5_2_model):
    response = client.chat.completions.create(
      model=model,
      messages=[
        {
          "role": "system",
          "content": "You are a business analyst collecting requirements for a software application. Your job is to review interview transcripts between an interviewer who is a business analyst and a stakeholder who is a prospective user of the application. The interviewer will ask the stakeholder questions to identify their requirements for the application."
        },
        {
          "role": "user",
          "content": prompt
        }
      ],
      response_format= {"type": "json_object"}
    )
    #print(response)
    return response.choices[0].message.content

In [3]:
import json

# data_path = 'data1'
data_path = 'data1_gpt52'

data = json.load(open('%s/transcripts.json' % data_path, 'r'))
print('Read %i transcripts' % len(data['transcript']))

Read 34 transcripts


In [4]:
sample_size = 10

In [5]:
prompt1 = """Read the following interview transcript excerpt and respond with any goals that the speaker expresses.
Write each goal in general language so that it describes only one action, and do not include references to applications,
products or services in the goal. Only write goals that can be traced to specific phrases in the speech.
Respond with the goals in a JSON list of strings.

%s

Goals: 
"""

prompt2 = """Read the following goals in JSON format and identify the substrings in the interview transcript
excerpt from which the goals were generated. Respond in JSON using the format
{'goal': 'goal statement', 'phrases': ['phrase1', 'phrase2']}

%s

%s

Response: """

def extract_transcript_goals(transcript):
    """
    Prompt the model to extract goals from a transcript
    :param list(dict) transcript: List containing speaker and text rows
    """
    extracted = []
    excerpt = '\n'.join(['%s: %s\n' % (t['speaker'], t['text']) for t in transcript])
    p1 = prompt1 % excerpt
    #print(p1)
    r1 = prompt_model(p1)
    #print(r1)

    goals = []
    phrases = []
    if r1:
        goals = json.loads(r1)

        p2 = prompt2 % (r1, excerpt)
        r2 = prompt_model(p2)
        #print(r2)
        if r2:
            phrases = json.loads(r2)

    return {'excerpt': excerpt, 'goals': goals, 'phrases': phrases} 


In [6]:
def one_extract_goal_request(data, i):
    e = extract_transcript_goals(data['transcript'][i])
    return e

In [7]:
def extract_and_add_goal(data, i, results):
    e = one_extract_goal_request(data, i)
    results[data['ids'][i]].append(e)

In [8]:
def extract_goals(data, i, results, count=10):
    print('Extracting goals from transcript %i with %i turns' % (i, len(data['transcript'][i])), end='')
    
    results[data['ids'][i]] = []
    j = 0
    while j < count: # goal extraction performed count=10 times
        print(' .', end='')
        extract_and_add_goal(data, i, results)
        j += 1
    print(' .done!')

In [9]:
# e = extract_transcript_goals(data['transcript'][0])

In [10]:
#results = {}
results = json.load(open('%s/extracted-whole.json' % data_path))

In [ ]:
i = 0
while i < len(data['ids']):
    if data['ids'][i] not in results:
        print(f"index {i} missing... extracting...")
        extract_goals(data, i, results, sample_size)

In [24]:
len(results[data['ids'][21]])

0

In [42]:
extract_goals(data, 21, results)

Extracting goals from transcript 21 with 33 turns . . . . . . . . . . .done!


In [43]:
extract_goals(data, 22, results)

Extracting goals from transcript 22 with 46 turns . . . . . . . . . . .done!


In [14]:
extract_goals(data, 23, results)

Extracting goals from transcript 23 with 36 turns . . . . . . . . . . .done!


In [15]:
extract_goals(data, 24, results)

Extracting goals from transcript 24 with 19 turns . . . . . . . . . . .done!


In [19]:
extract_goals(data, 25, results)

Extracting goals from transcript 25 with 67 turns . . . . . . . . . . .done!


In [20]:
extract_goals(data, 26, results)

Extracting goals from transcript 26 with 60 turns . . . . . . . . . . .done!


In [21]:
extract_goals(data, 27, results)

Extracting goals from transcript 27 with 163 turns . . . . . . . . . . .done!


In [22]:
extract_goals(data, 28, results)

Extracting goals from transcript 28 with 142 turns . . . . . . . . . . .done!


In [25]:
extract_goals(data, 29, results)

Extracting goals from transcript 29 with 50 turns . . . . . . . . . . .done!


In [26]:
extract_goals(data, 30, results)

Extracting goals from transcript 30 with 116 turns . . . . . . . . . . .done!


In [27]:
extract_goals(data, 31, results)

Extracting goals from transcript 31 with 75 turns . . . . . . . . . . .done!


In [30]:
extract_goals(data, 32, results)

Extracting goals from transcript 32 with 118 turns . . . . . . . . . . .done!


In [31]:
extract_goals(data, 33, results)

Extracting goals from transcript 33 with 27 turns . . . . . . . . . . .done!


In [50]:
data['ids'][0]

'32'

In [13]:
data['ids'][22] in results

True

In [37]:
i = 0
while i < len(data['ids']):
    if data['ids'][i] not in results:
        print(f"index {i} missing... extracting...")
        extract_goals(data, i, results, sample_size)
    else:
        print(f"index {i} found... skipping")
    while len(results[data['ids'][i]]) < sample_size:
        print(f"len(results[data['ids'][{i}]]) == {len(results[data['ids'][i]])} < {sample_size}")
        extract_and_add_goal(data, i, results)
        break
    i += 1

index 0 missing... extracting...
Extracting goals from transcript 0 with 84 turns . . . . . . . . . . .done!
index 1 missing... extracting...
Extracting goals from transcript 1 with 50 turns . . . . . . . . . . .done!
index 2 missing... extracting...
Extracting goals from transcript 2 with 27 turns . . . . . . . . . . .done!
index 3 missing... extracting...
Extracting goals from transcript 3 with 191 turns . . . . . . . . . . .done!
index 4 missing... extracting...
Extracting goals from transcript 4 with 104 turns . . . . . . . . . . .done!
index 5 missing... extracting...
Extracting goals from transcript 5 with 70 turns . . . . . . . . . . .done!
index 6 missing... extracting...
Extracting goals from transcript 6 with 111 turns . . . . . . . . . . .done!
index 7 missing... extracting...
Extracting goals from transcript 7 with 59 turns . . . . . . . . . . .done!
index 8 missing... extracting...
Extracting goals from transcript 8 with 111 turns . . . . . . . . . . .done!
index 9 missing

In [38]:
json.dump(results, open('%s/extracted-whole.json' % data_path, 'w'))

In [32]:
print(results.keys())

dict_keys(['17', '10', '19', '26', '8', '21', '36', '31', '30', '24', '23', '4', '15', '3', '12', '13', '5', '14', '22', '25'])


In [41]:
for i in results:
    print(len(results[i]))

10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
